# PPO LunarLander — Deep RL Course (Unit 1)

Trains a PPO agent to solve LunarLander using Stable-Baselines3, then pushes it to the Hugging Face Hub.

Note: this notebook uses `LunarLander-v2` (registered as an alias of v3's physics) because the Deep RL Course certification checker looks for the exact string `LunarLander-v2`.

## 1. Install dependencies

In [1]:
!apt install -y swig cmake
!pip install --only-binary :all: "pygame>=2.6.0"
!pip install stable-baselines3 huggingface_sb3 "gymnasium[box2d]" swig shimmy --upgrade -q

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
swig is already the newest version (4.2.0-2ubuntu1).
cmake is already the newest version (3.28.3-1build7).
0 upgraded, 0 newly installed, 0 to remove and 44 not upgraded.


## 2. Set up rendering + virtual display (needed for video recording on Colab)

In [2]:
!sudo apt-get update -y
!sudo apt-get install -y python3-opengl ffmpeg xvfb
!pip install pyvirtualdisplay -q

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:3 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:4 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:5 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:7 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
python3-opengl is already the newest version (3.1.7+dfsg-1).
ffmpeg is already the newest version (7:6.1.1-3ubuntu5).
xvfb is already the newest version (2:21.1.12-1ubunt

In [ ]:
import os
os.kill(os.getpid(), 9)  # restart runtime so newly installed packages load cleanly

## 3. Start virtual display
Run this cell right after the runtime restarts.

In [3]:
from pyvirtualdisplay import Display
virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

## 4. Imports

In [4]:
import gymnasium as gym
from gymnasium.envs.registration import register

from huggingface_sb3 import load_from_hub, package_to_hub
from huggingface_hub import notebook_login

from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 5. Register `LunarLander-v2`
Newer Gymnasium versions dropped `v2`. This registers `v2` as an alias pointing to the same LunarLander implementation, so the environment name matches what the course's certification checker expects.

In [5]:
register(
    id="LunarLander-v2",
    entry_point="gymnasium.envs.box2d.lunar_lander:LunarLander",
    max_episode_steps=1000,
    reward_threshold=200,
)

## 6. Explore the environment (optional, for understanding)

In [6]:
env = gym.make("LunarLander-v2")
env.reset()

print("_____OBSERVATION SPACE_____")
print("Observation Space Shape", env.observation_space.shape)
print("Sample observation", env.observation_space.sample())

print("\n_____ACTION SPACE_____")
print("Action Space Shape", env.action_space.n)
print("Action Space Sample", env.action_space.sample())

_____OBSERVATION SPACE_____
Observation Space Shape (8,)
Sample observation [ 0.03740554  0.19907509 -4.9448633   8.31517     2.6586301   3.3862906
  0.18570317  0.41751072]

_____ACTION SPACE_____
Action Space Shape 4
Action Space Sample 3


/usr/local/lib/python3.13/dist-packages/gymnasium/envs/registration.py:513: DeprecationWarning: WARN: The environment LunarLander-v2 is out of date. You should consider upgrading to version `v3`.
  logger.deprecation(
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type swigvarlink has no __module__ attribute


## 7. Create 16 parallel training environments

In [7]:
env = make_vec_env("LunarLander-v2", n_envs=16)

## 8. Define the PPO model

In [8]:
model = PPO(
    policy="MlpPolicy",
    env=env,
    n_steps=2048,
    batch_size=128,
    n_epochs=10,
    gamma=0.999,
    gae_lambda=0.98,
    ent_coef=0.01,
    learning_rate=3e-4,
    clip_range=0.2,
    verbose=1,
)

Using cpu device


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 9. Train

In [9]:
model.learn(total_timesteps=2_000_000)

model_name = "ppo-LunarLander-v2"
model.save(model_name)

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 92.8     |
|    ep_rew_mean     | -195     |
| time/              |          |
|    fps             | 5696     |
|    iterations      | 1        |
|    time_elapsed    | 5        |
|    total_timesteps | 32768    |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 99.2         |
|    ep_rew_mean          | -143         |
| time/                   |              |
|    fps                  | 3697         |
|    iterations           | 2            |
|    time_elapsed         | 17           |
|    total_timesteps      | 65536        |
| train/                  |              |
|    approx_kl            | 0.0073667187 |
|    clip_fraction        | 0.0614       |
|    clip_range           | 0.2          |
|    entropy_loss         | -1.38        |
|    explained_variance   | 0.00171      |
|    learning_r

## 10. Evaluate
Use enough episodes to get a stable estimate (variance can be high with too few).

In [10]:
eval_env = Monitor(gym.make("LunarLander-v2", render_mode="rgb_array"))
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=20, deterministic=True)
print(f"mean_reward={mean_reward:.2f} +/- {std_reward}")

/usr/local/lib/python3.13/dist-packages/gymnasium/envs/registration.py:513: DeprecationWarning: WARN: The environment LunarLander-v2 is out of date. You should consider upgrading to version `v3`.
  logger.deprecation(


mean_reward=287.04 +/- 15.501714903461927


## 11. Log in to Hugging Face

In [11]:
notebook_login()
!git config --global credential.helper store

## 12. Push model, results, and model card to the Hub

In [13]:
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [14]:
repo_id = "Ebishj/ppo-LunarLander-v2-clean"  # new repo
env_id = "LunarLander-v2"              # must match this exact string for certification
model_architecture = "PPO"
commit_message = "Upload PPO LunarLander agent"

eval_env = DummyVecEnv([lambda: Monitor(gym.make(env_id, render_mode="rgb_array"))])

package_to_hub(
    model=model,
    model_name=model_name,
    model_architecture=model_architecture,
    env_id=env_id,
    eval_env=eval_env,
    repo_id=repo_id,
    commit_message=commit_message,
)

/usr/local/lib/python3.13/dist-packages/gymnasium/envs/registration.py:513: DeprecationWarning: WARN: The environment LunarLander-v2 is out of date. You should consider upgrading to version `v3`.
  logger.deprecation(


ℹ This function will save, evaluate, generate a video of your agent,
create a model card and push everything to the hub. It might take up to 1min.
This is a work in progress: if you encounter a bug, please open an issue.
Saving video to /tmp/tmp6c55slwq/-step-0-to-step-1000.mp4


/usr/local/lib/python3.13/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Moviepy - Building video /tmp/tmp6c55slwq/-step-0-to-step-1000.mp4.
Moviepy - Writing video /tmp/tmp6c55slwq/-step-0-to-step-1000.mp4



Moviepy - Done !
Moviepy - video ready /tmp/tmp6c55slwq/-step-0-to-step-1000.mp4
✘ 'DummyVecEnv' object has no attribute 'video_recorder'
✘ We are unable to generate a replay of your agent, the package_to_hub
process continues
✘ Please open an issue at
https://github.com/huggingface/huggingface_sb3/issues
ℹ Pushing repo Ebishj/ppo-LunarLander-v2-clean to the Hugging Face
Hub


...

  ...c4/ppo-LunarLander-v2.zip: 100%|##########|  150kB /  150kB            

  ...LunarLander-v2/policy.pth: 100%|##########| 44.0kB / 44.0kB            

  ...r-v2/policy.optimizer.pth: 100%|##########| 88.4kB / 88.4kB            

ℹ Your model is pushed to the Hub. You can view your model here:
https://huggingface.co/Ebishj/ppo-LunarLander-v2-clean/tree/main/


CommitInfo(commit_url='https://huggingface.co/Ebishj/ppo-LunarLander-v2-clean/commit/37179e14b097046b7d950c8fcc3766ae6a06de6b', commit_message='Upload PPO LunarLander agent', commit_description='', oid='37179e14b097046b7d950c8fcc3766ae6a06de6b', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Ebishj/ppo-LunarLander-v2-clean', endpoint='https://huggingface.co', repo_type='model', repo_id='Ebishj/ppo-LunarLander-v2-clean'), pr_revision=None, pr_num=None)

## 13. Record and upload a replay video manually
`package_to_hub`'s built-in video generation has a known bug (`DummyVecEnv has no attribute video_recorder`), so we record and upload it separately.

In [15]:
from stable_baselines3.common.vec_env import VecVideoRecorder

video_env = DummyVecEnv([lambda: gym.make("LunarLander-v2", render_mode="rgb_array")])
video_env = VecVideoRecorder(
    video_env,
    "/content/video",
    record_video_trigger=lambda x: x == 0,
    video_length=1000,
    name_prefix="replay",
)

obs = video_env.reset()
for _ in range(1000):
    action, _ = model.predict(obs, deterministic=True)
    obs, _, _, _ = video_env.step(action)
video_env.close()

/usr/local/lib/python3.13/dist-packages/gymnasium/envs/registration.py:513: DeprecationWarning: WARN: The environment LunarLander-v2 is out of date. You should consider upgrading to version `v3`.
  logger.deprecation(


Saving video to /content/video/replay-step-0-to-step-1000.mp4
Moviepy - Building video /content/video/replay-step-0-to-step-1000.mp4.
Moviepy - Writing video /content/video/replay-step-0-to-step-1000.mp4



Moviepy - Done !
Moviepy - video ready /content/video/replay-step-0-to-step-1000.mp4


In [16]:
import glob
from huggingface_hub import upload_file

video_path = glob.glob("/content/video/*.mp4")[0]

upload_file(
    path_or_fileobj=video_path,
    path_in_repo="replay.mp4",
    repo_id=repo_id,
)

print(f"Uploaded {video_path} to {repo_id}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...y-step-0-to-step-1000.mp4: 100%|##########|  162kB /  162kB            

Uploaded /content/video/replay-step-0-to-step-1000.mp4 to Ebishj/ppo-LunarLander-v2-clean


## Done

Check your result at the certification tracker: https://huggingface.co/spaces/huggingface-projects/Deep-RL-Course-Certification

Your model is live at: `https://huggingface.co/<repo_id>`